In [1]:
import openai
import instructor
from qdrant_client import QdrantClient

from pydantic import BaseModel, Field


In [2]:
prompt = """
You are a helpful assistant.
Return an answer to the question.
Question: What is your name?
"""


In [3]:
response = openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    temperature=0
)

print(response.choices[0].message.content)


I am an AI language model and don't have a personal name, but you can call me Assistant! How can I help you today?


### Adding Instructor

In [4]:
client = instructor.from_openai(openai.OpenAI())


In [5]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")


In [6]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    temperature=0,
    response_model=RAGGenerationResponse
)

print(response)


answer='I am an AI assistant and do not have a personal name like a human. You can call me Assistant!'


In [7]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")
    reasoning: str = Field(description="The reasoning for the answer")


In [8]:
response, raw_response = client.chat.completions.create_with_completion(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    temperature=0,
    response_model=RAGGenerationResponse
)


In [9]:
response

RAGGenerationResponse(answer='I am an AI assistant and do not have a personal name like a human. You can call me Assistant.', reasoning="As an AI, I don't possess a personal identity or name. I am designed to assist users with their queries.")

In [10]:
raw_response

ChatCompletion(id='chatcmpl-Dg4DajTkd5m1Rzebw4RHUdgJzhpmI', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_QXFRzdPTiJbdNWbXnbq8gBYb', function=Function(arguments='{"answer":"I am an AI assistant and do not have a personal name like a human. You can call me Assistant.","reasoning":"As an AI, I don\'t possess a personal identity or name. I am designed to assist users with their queries."}', name='RAGGenerationResponse'), type='function')]))], created=1778918362, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_0e72352265', usage=CompletionUsage(completion_tokens=52, prompt_tokens=110, total_tokens=162, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction

### RAG Example

In [11]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")


In [15]:
from dotenv import load_dotenv
from groq import Groq
from qdrant_client import QdrantClient
from openai import OpenAI
import os


load_dotenv("../.env", override=True)

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
      api_key=os.environ.get("OPENROUTER_API_KEY"),

)
qdrant_client = QdrantClient(host="localhost", port=6333)

groq_client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)



import logging

logger = logging.getLogger(__name__)


def get_embedding(text: str, model="openai/text-embedding-3-small"):
    response = openrouter_client.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

def retrieve_products(query: str, limit: int = 3):
    """
    Search for products based on a natural language query.
    """
    # 1. Convert the user's text query into a vector embedding
    query_vector = get_embedding(query)
    
    # 2. Search the Qdrant database for the closest matching vectors
    search_results = qdrant_client.query_points(
        collection_name="products",
        query=query_vector,
        limit=limit
    )


    retrieve_context = []
    average_rating = []
    similarity_score = []
    context_id = []

    for result in search_results.points:
        # Extract fields from the payload and the Qdrant result object
        retrieve_context.append(result.payload["description"])
        average_rating.append(result.payload["average_rating"])
        similarity_score.append(result.score)
        context_id.append(result.payload["parent_asin"])
        
    return {
        "retrieved_context": retrieve_context,
        "retrieved_context_ratings": average_rating,
        "similarity_score": similarity_score,
        "retrieved_context_ids": context_id
    }




def process_context(context):
    formatted_context = ""
    
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
        
    return formatted_context



def build_prompt(preprocessed_context, question):
        prompt = f"""
        You are a shopping assistant that can answer questions about the products in stock.

        You will be given a question and a list of context.

        Instructions:
        - You need to answer the question based on the provided context only.
        - Never use word context and refer to it as the available products.

        Context:
        {preprocessed_context}

        Question:
        {question}
        """
        return prompt



def generate_answer(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        response_model=RAGGenerationResponse
    )

    return response



def rag_pipeline(question: str, top_k: int = 5) -> str:
    try:
        retrieved_context = retrieve_products(question, limit=top_k)
        preprocessed_context = process_context(retrieved_context)
        prompt = build_prompt(preprocessed_context, question)
        answer = generate_answer(prompt)


        final_result={
            "data_model":answer,
            "answer":answer.answer,
            "question":question,
            "retrieved_context":retrieved_context["retrieved_context"],
            "average_rating":retrieved_context["retrieved_context_ratings"],
            "similarity_score": retrieved_context["similarity_score"],
            "context_id":retrieved_context["retrieved_context_ids"]
        }


        return final_result
    except Exception as e:
        logger.error(f"Error in RAG pipeline: {e}")
        return {
            "answer": "Sorry, there was an error processing your request.",
            "question": question,
            "retrieved_context": [],
            "average_rating": [],
            "similarity_score": [],
            "context_id": []
        }




In [18]:
output = rag_pipeline("I need something to moisturize my skin, what do you recommend?")

In [20]:
print(output["answer"])

I recommend the 'Island Spice' Magnesium Cream (ID: B00F97Y8RA), which has a light texture and is enriched with rich plant oils and butters to nurture and protect your skin, leaving it silky smooth. Additionally, the Bee Venom product (ID: B07S2HKRVM) is also a great option as it provides intense hydration and is suitable for all skin types.
